# Step 3 — Robustness / distribution shift (ResNet50, 20 models)

Course component 4: *evaluate under a change the dataset supports.* Cassava has no site, device or
date labels, so we study two kinds of shift:

**A. Synthetic capture degradations.** The 20 grouped-split models (4 methods × 5 folds) are
re-scored on corrupted copies of the test set; nothing is retrained. Six corruptions a phone photo
taken in a field can suffer, each at 3 severities:

| corruption | severity 1 / 2 / 3 | real-world cause |
|---|---|---|
| blur | Gaussian σ = 1 / 2 / 4 px | out of focus, hand shake |
| dark | brightness ×0.6 / 0.4 / 0.25 | shade, evening |
| bright | brightness ×1.4 / 1.7 / 2.0 | direct sun, overexposure |
| noise | Gaussian, std 10 / 20 / 35 | low light, cheap sensor |
| jpeg | quality 30 / 15 / 7 | compression when photos are sent |
| lowres | 192 / 96 / 64 px, upsampled back | low-resolution camera, digital zoom |

Scores come from `Deep_Learning/evaluation/robustness.py` (run once, GPU), which first checked that
its clean predictions reproduce each run's saved `test_probs.npy`.

**B. Natural capture conditions.** The clean test photos already vary in exposure and sharpness.
We bucket them by measured brightness and sharpness and compare performance across buckets.

Sections: 1 Setup · 2 What the corruptions look like · 3 Results table · 4 Degradation curves ·
5 Which method is most robust? · 6 Which classes break, and into what? · 7 Natural capture conditions · 8 Summary

Outputs: `reports/figures/robustness/`.

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage, stats
from sklearn.metrics import balanced_accuracy_score, f1_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Deep_Learning").is_dir())
sys.path.insert(0, str(ROOT))
from Deep_Learning.config import SPLITS_CSV, short_class_names
from Deep_Learning.evaluation.robustness import SEVERITIES, corrupt

CLASSES = short_class_names()
K = len(CLASSES)
EVAL = ROOT / "reports" / "robustness_eval"          # written by robustness.py
OUT = ROOT / "reports" / "figures" / "robustness"
OUT.mkdir(parents=True, exist_ok=True)

BASELINE = "none"
METHODS = ["none", "sampler", "weights", "oversample"]
OTHERS = [m for m in METHODS if m != BASELINE]
LABEL = {"none": "None (baseline)", "sampler": "Weighted sampler",
         "weights": "Class-weighted loss", "oversample": "Oversampling"}
COLOR = {"none": "#2a78d6", "sampler": "#eb6834", "weights": "#1baf7a", "oversample": "#eda100"}
CORR = list(SEVERITIES)
CNAME = {"blur": "Blur", "dark": "Darker", "bright": "Brighter", "noise": "Noise", "jpeg": "JPEG", "lowres": "Low resolution"}
SURFACE, INK, INK_2, GRID = "#ffffff", "#0b0b0b", "#52514e", "#e4e3df"
HALO = dict(boxstyle="square,pad=0.15", fc=SURFACE, ec="none")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
    "text.color": INK, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 80,
})
pd.set_option("display.precision", 3)

def save(fig, stem, pdf=True):
    fig.savefig(OUT / f"{stem}.png", dpi=250, bbox_inches="tight")
    if pdf:
        fig.savefig(OUT / f"{stem}.pdf", bbox_inches="tight")

def clean(ax, grid="y"):
    getattr(ax, f"{grid}axis").grid(True, color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)

test = pd.read_csv(SPLITS_CSV).query("split == 'test'").reset_index(drop=True)
y = test.label.to_numpy()
m = pd.read_csv(EVAL / "metrics.csv")
m["variant"] = np.where(m.corruption == "clean", "clean", m.corruption + "_s" + m.severity.astype(str))
print(len(m), "rows:", m.run.nunique(), "runs x", m.variant.nunique(), "variants")

380 rows: 20 runs x 19 variants


## 2. What the corruptions look like

One test image (center crop, as the model sees it) under every corruption and severity.

In [2]:
X = np.load(EVAL / "test_uint8_384.npy", mmap_mode="r")
idx = 7
x0 = np.array(X[idx:idx + 1])   # writable copy
fig, axes = plt.subplots(len(CORR), 4, figsize=(7.2, 1.9 * len(CORR)))
for r, kind in enumerate(CORR):
    for c in range(4):
        img = x0 if c == 0 else corrupt(x0, kind, SEVERITIES[kind][c - 1])
        axes[r, c].imshow(img[0].transpose(1, 2, 0))
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        for s in axes[r, c].spines.values():
            s.set_visible(False)
        if r == 0:
            axes[r, c].set_title("clean" if c == 0 else f"severity {c}", fontsize=9)
    axes[r, 0].set_ylabel(CNAME[kind], fontsize=9)
fig.tight_layout()
save(fig, "corruption_examples", pdf=False)
plt.show()

## 3. Results table

Macro-F1 (%) per corruption and severity: mean ± std over the 5 folds of each method.

In [3]:
agg = m.groupby(["corruption", "severity", "method"]).macro_f1.agg(["mean", "std"]).reset_index()
order = [("clean", 0)] + [(c, s) for c in CORR for s in (1, 2, 3)]
tab = pd.DataFrame({LABEL[me]: [f"{100*agg.query('corruption==@c and severity==@s and method==@me')['mean'].item():.1f} "
                                f"± {100*agg.query('corruption==@c and severity==@s and method==@me')['std'].item():.1f}"
                                for c, s in order] for me in METHODS},
                   index=pd.MultiIndex.from_tuples(order, names=["corruption", "severity"]))
tab.to_csv(OUT / "robustness_macro_f1.csv")
tab

None (baseline) Weighted sampler Class-weighted loss  \
corruption severity                                                        
clean      0             77.7 ± 0.7       77.0 ± 0.9          78.3 ± 0.4   
blur       1             74.6 ± 0.4       73.4 ± 1.1          74.3 ± 1.2   
           2             65.0 ± 2.7       59.5 ± 4.3          64.5 ± 2.2   
           3             39.0 ± 9.7       29.2 ± 7.9          40.2 ± 4.0   
dark       1             77.7 ± 0.9       76.9 ± 0.6          77.6 ± 0.6   
           2             76.7 ± 1.0       76.0 ± 0.4          76.4 ± 0.9   
           3             74.6 ± 1.0       74.2 ± 0.4          73.9 ± 0.9   
bright     1             75.9 ± 0.5       74.5 ± 0.8          75.3 ± 0.9   
           2             72.5 ± 0.6       71.5 ± 0.6          71.8 ± 1.0   
           3             68.1 ± 0.8       66.8 ± 1.5          67.0 ± 1.7   
noise      1             71.7 ± 1.8       71.6 ± 1.3          72.9 ± 1.4   
           2             53.3 ± 4.3       52.1 ± 3.8          53.4 ± 4.0   
           3             32.5 ± 6.8       29.0 ± 4.9          29.8 ± 4.2   
jpeg       1             73.9 ± 1.2       72.8 ± 0.9          74.1 ± 1.3   
           2             67.2 ± 1.6       66.6 ± 1.1          68.5 ± 1.2   
           3             50.0 ± 4.3       54.0 ± 2.2          53.7 ± 2.8   
lowres     1             72.6 ± 1.0       70.5 ± 1.1          72.2 ± 1.7   
           2             60.7 ± 2.9       53.3 ± 5.2          60.8 ± 1.9   
           3             46.1 ± 4.6       36.0 ± 6.8          44.9 ± 3.0   

                    Oversampling  
corruption severity               
clean      0          77.7 ± 1.0  
blur       1          74.6 ± 1.6  
           2          63.7 ± 1.4  
           3          42.0 ± 3.7  
dark       1          77.5 ± 0.7  
           2          76.6 ± 0.8  
           3          74.4 ± 1.0  
bright     1          76.2 ± 1.0  
           2          72.2 ± 1.7  
           3          67.1 ± 1.5  
noise      1          72.1 ± 1.2  
           2          48.1 ± 7.6  
           3         26.3 ± 10.4  
jpeg       1          73.1 ± 0.6  
           2          67.7 ± 1.6  
           3          53.8 ± 3.7  
lowres     1          71.8 ± 1.4  
           2          58.8 ± 0.9  
           3          44.9 ± 3.3

## 4. Degradation curves

Macro-F1 against severity (0 = clean), mean over the 5 folds of each method; shaded band = ± 1 std.
The dashed line is chance level for macro-F1 when always predicting CMD (≈ 0.15).

In [4]:
fig, axes = plt.subplots(1, len(CORR), figsize=(13, 2.9), sharey=True)
for ax, kind in zip(axes, CORR):
    for me in METHODS:
        d = pd.concat([agg.query("corruption=='clean' and method==@me"),
                       agg.query("corruption==@kind and method==@me")]).sort_values("severity")
        ax.plot(d.severity, 100 * d["mean"], "o-", ms=4, lw=1.8, color=COLOR[me], label=LABEL[me])
        ax.fill_between(d.severity, 100 * (d["mean"] - d["std"]), 100 * (d["mean"] + d["std"]),
                        color=COLOR[me], alpha=0.15, lw=0)
    ax.axhline(100 * 2 * 0.61 / 1.61 / 5, color=INK_2, ls="--", lw=0.8)
    ax.set_title(CNAME[kind], fontsize=9)
    ax.set_xticks([0, 1, 2, 3])
    ax.set_xlabel("severity")
    ax.set_ylim(0, 85)
    clean(ax)
axes[0].set_ylabel("macro-F1 (%)")
fig.legend(*axes[0].get_legend_handles_labels(), ncol=4, frameon=False, fontsize=8, loc="lower center",
           bbox_to_anchor=(0.5, 0.98))
fig.tight_layout()
save(fig, "degradation_curves")
plt.show()

## 5. Which method is most robust?

For every run: **relative drop** = 1 − (corrupted macro-F1 / clean macro-F1), averaged over all
6 corruptions × 3 severities (similar in spirit to the mean corruption error of ImageNet-C).
Then the paired comparison with the baseline fold by fold, as in step 2.

In [5]:
clean_f1 = m[m.corruption == "clean"].set_index("run").macro_f1
mc = m[m.corruption != "clean"].copy()
mc["rel_drop"] = 1 - mc.macro_f1 / mc.run.map(clean_f1)
per_run = mc.groupby(["run", "method", "fold"]).agg(mean_corrupted_f1=("macro_f1", "mean"),
                                                    mean_rel_drop=("rel_drop", "mean")).reset_index()
per_run["clean_f1"] = per_run.run.map(clean_f1)
rob = per_run.groupby("method")[["clean_f1", "mean_corrupted_f1", "mean_rel_drop"]].agg(["mean", "std"]).loc[METHODS]
rob.to_csv(OUT / "robustness_summary.csv", float_format="%.4f")
(100 * rob).round(1)

clean_f1      mean_corrupted_f1      mean_rel_drop     
               mean  std              mean  std          mean  std
method                                                            
none           77.7  0.7              64.0  1.0          17.6  1.8
sampler        77.0  0.9              61.6  1.6          20.0  2.5
weights        78.3  0.4              64.0  0.7          18.3  0.8
oversample     77.7  1.0              63.4  1.2          18.4  2.5

In [6]:
rows = []
for k in ["mean_corrupted_f1", "mean_rel_drop"]:
    piv = per_run.pivot(index="method", columns="fold", values=k)
    for me in OTHERS:
        d = (piv.loc[me] - piv.loc[BASELINE]).to_numpy()
        better = (d < 0).sum() if k == "mean_rel_drop" else (d > 0).sum()
        rows.append({"metric": k, "method": me, "mean_diff_pp": 100 * d.mean(), "std_pp": 100 * d.std(ddof=1),
                     "folds_better": f"{better}/{len(d)}",
                     "p_ttest": stats.ttest_rel(piv.loc[me], piv.loc[BASELINE]).pvalue})
rob_paired = pd.DataFrame(rows)

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty_like(p); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

rob_paired["p_holm"] = rob_paired.groupby("metric").p_ttest.transform(holm)   # 3 methods vs baseline
rob_paired.to_csv(OUT / "robustness_paired.csv", index=False, float_format="%.4g")
rob_paired.round(3)

,metric,method,mean_diff_pp,std_pp,folds_better,p_ttest,p_holm
0,mean_corrupted_f1,sampler,-2.434,1.751,0/5,0.036,0.108
1,mean_corrupted_f1,weights,-0.026,0.922,2/5,0.954,0.954
2,mean_corrupted_f1,oversample,-0.598,0.748,1/5,0.149,0.297
3,mean_rel_drop,sampler,2.447,2.671,0/5,0.110,0.330
4,mean_rel_drop,weights,0.722,1.517,1/5,0.347,0.516
5,mean_rel_drop,oversample,0.842,1.429,2/5,0.258,0.516


In [7]:
# Relative drop per corruption (mean over the 3 severities and the 5 folds), per method
heat = mc.groupby(["method", "corruption"]).rel_drop.mean().unstack()[CORR].loc[METHODS]
fig, ax = plt.subplots(figsize=(6.4, 2.4))
im = ax.imshow(100 * heat.to_numpy(), cmap="Blues", vmin=0, vmax=100 * heat.to_numpy().max())
for i in range(len(METHODS)):
    for j in range(len(CORR)):
        v = 100 * heat.iloc[i, j]
        ax.text(j, i, f"{v:.0f}%", ha="center", va="center", fontsize=8,
                color="white" if v > 0.55 * 100 * heat.to_numpy().max() else "black")
ax.set_xticks(range(len(CORR)), [CNAME[c] for c in CORR])
ax.set_yticks(range(len(METHODS)), [LABEL[me] for me in METHODS])
ax.set_title("Relative macro-F1 drop (mean over 3 severities, 5 folds)", fontsize=9)
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
save(fig, "relative_drop_heatmap")
plt.show()

## 6. Which classes break, and into what?

Per-class recall under each corruption at **severity 2** (mean over all 20 models), and the share of
test images predicted as each class: if a corruption pushes everything into one class, the model has
learned a shortcut (e.g. "yellow-ish → CBSD", "dark → CMD").

In [8]:
s2 = m[(m.severity == 2) | (m.corruption == "clean")]
rec = s2.groupby("variant")[[f"recall_{c}" for c in range(K)]].mean()
rec.columns = CLASSES
rec = rec.loc[["clean"] + [f"{c}_s2" for c in CORR]]
(100 * rec).round(1)

,CBB,CBSD,CGM,CMD,Healthy
variant,,,,,
clean,63.6,78.2,77.3,95.5,75.7
blur_s2,58.8,56.2,39.7,90.6,73.5
dark_s2,60.7,76.7,74.1,94.8,77.5
bright_s2,51.8,75.4,66.7,93.1,73.8
noise_s2,22.7,38.7,90.1,75.2,43.2
jpeg_s2,59.8,47.6,64.3,93.5,72.2
lowres_s2,58.5,46.5,37.0,88.8,64.8


In [9]:
P = np.load(EVAL / "probs.npz")
runs = sorted(m.run.unique())
share = {}
for v in ["clean"] + [f"{c}_s{s}" for c in CORR for s in (2, 3)]:
    preds = np.concatenate([P[f"{r}__{v}"].astype(np.float32).argmax(1) for r in runs])
    share[v] = np.bincount(preds, minlength=K) / len(preds)
share = pd.DataFrame(share, index=CLASSES).T
share.loc["true distribution"] = np.bincount(y, minlength=K) / len(y)
(100 * share).round(1)

,CBB,CBSD,CGM,CMD,Healthy
clean,5.3,9.8,10.9,60.7,13.3
blur_s2,6.4,8.2,6.2,60.4,18.9
blur_s3,18.8,8.2,9.7,54.9,8.5
dark_s2,5.0,10.0,10.5,60.4,14.2
dark_s3,4.7,10.9,10.5,59.6,14.3
bright_s2,4.3,10.7,10.2,60.0,14.8
bright_s3,3.6,11.1,11.5,58.2,15.7
noise_s2,2.0,4.9,35.8,47.7,9.6
noise_s3,1.3,3.9,60.7,22.6,11.5
jpeg_s2,6.1,5.7,10.6,62.1,15.4


## 7. Natural capture conditions (no synthetic corruption)

Measured on the clean test images, as the model sees them (384×384 center crop):
- **brightness** = mean luminance (0–255);
- **sharpness** = variance of the Laplacian of the grey image (low = blurry).

Each is split into three equal-size buckets (tertiles). We report macro-F1 and balanced accuracy per
bucket (mean ± std over the 5 folds) and the class mix of each bucket, because a bucket that
contains more of a hard class will score lower for that reason alone.

In [10]:
lum_w = np.array([0.299, 0.587, 0.114], np.float32)
bright, sharp = np.empty(len(X)), np.empty(len(X))
for i in range(0, len(X), 256):
    g = np.tensordot(np.asarray(X[i:i + 256], np.float32), lum_w, axes=([1], [0]))   # (b, H, W)
    bright[i:i + 256] = g.mean((1, 2))
    sharp[i:i + 256] = np.array([ndimage.laplace(im).var() for im in g])
test["brightness"], test["sharpness"] = bright, sharp
test["brightness bucket"] = pd.qcut(bright, 3, labels=["dark", "medium", "bright"])
test["sharpness bucket"] = pd.qcut(sharp, 3, labels=["blurry", "medium", "sharp"])
test[["brightness", "sharpness"]].describe().round(1)

,brightness,sharpness
count,3078.0,3078.0
mean,119.5,1498.6
std,19.0,986.9
min,54.5,15.5
25%,104.8,735.6
50%,119.4,1293.7
75%,135.3,2086.0
max,180.7,5364.7


In [11]:
clean_probs = {r: P[f"{r}__clean"].astype(np.float32) for r in runs}
run_method = m.drop_duplicates("run").set_index("run")[["method", "fold"]]

def bucket_scores(col):
    rows = []
    for r in runs:
        pr = clean_probs[r].argmax(1)
        for b in test[col].cat.categories:
            sel = (test[col] == b).to_numpy()
            rows.append({"method": run_method.loc[r, "method"], "fold": run_method.loc[r, "fold"], "bucket": b,
                         "macro_f1": f1_score(y[sel], pr[sel], average="macro"),
                         "balanced_acc": balanced_accuracy_score(y[sel], pr[sel])})
    return pd.DataFrame(rows)

nat = {col: bucket_scores(col) for col in ["brightness bucket", "sharpness bucket"]}
tables = []
for col, d in nat.items():
    t = d.groupby(["bucket", "method"], observed=True).macro_f1.agg(["mean", "std"]).unstack("method")
    t = t.reindex(columns=pd.MultiIndex.from_product([["mean", "std"], METHODS]))
    t.index = pd.MultiIndex.from_product([[col], t.index])
    tables.append(t)
natural = pd.concat(tables)
natural.to_csv(OUT / "natural_conditions_macro_f1.csv", float_format="%.4f")
(100 * natural["mean"]).round(1)

none  sampler  weights  oversample
                  bucket                                    
brightness bucket bright  79.6     79.1     79.6        79.2
                  dark    75.6     75.0     77.2        76.4
                  medium  76.7     76.0     76.9        76.4
sharpness bucket  blurry  79.2     78.3     79.5        79.1
                  medium  77.7     76.6     78.8        78.2
                  sharp   73.5     73.3     73.6        73.2

In [12]:
# Class mix of each bucket (%), to read the table above correctly
mix = pd.concat({col: pd.crosstab(test[col], test.label.map(dict(enumerate(CLASSES))), normalize="index")
                 for col in ["brightness bucket", "sharpness bucket"]})
(100 * mix[CLASSES]).round(1)

label                     CBB  CBSD   CGM   CMD  Healthy
brightness bucket dark    6.6  10.9  10.0  59.1     13.4
                  medium  4.2  10.1  11.8  62.8     11.1
                  bright  3.7   8.8  11.8  61.4     14.3
sharpness bucket  blurry  7.9  15.1  10.7  47.4     18.9
                  medium  4.1   9.9  10.8  63.8     11.3
                  sharp   2.5   4.8  12.1  72.0      8.6

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(9, 2.8), sharey=True)
for ax, (col, d) in zip(axes, nat.items()):
    cats = list(test[col].cat.categories)
    for j, me in enumerate(METHODS):
        g = d[d.method == me].groupby("bucket", observed=True).macro_f1.agg(["mean", "std"]).loc[cats]
        xs = np.arange(len(cats)) + (j - 1.5) * 0.12
        ax.errorbar(xs, 100 * g["mean"], yerr=100 * g["std"], fmt="o", ms=5, color=COLOR[me],
                    elinewidth=1.4, capsize=0, label=LABEL[me])
    ax.set_xticks(range(len(cats)), cats)
    ax.set_title(col.replace(" bucket", "").capitalize() + " of the test photo (tertiles)", fontsize=9)
    clean(ax)
axes[0].set_ylabel("macro-F1 (%)")
axes[1].legend(frameon=False, fontsize=7, loc="lower right")
fig.tight_layout()
save(fig, "natural_conditions")
plt.show()

## 8. Summary

Printed from the tables above.

In [14]:
print("Mean macro-F1 over all 20 models:")
print(f"  clean                 {100*m[m.corruption=='clean'].macro_f1.mean():.1f}")
for kind in CORR:
    vals = [100 * m[(m.corruption == kind) & (m.severity == s)].macro_f1.mean() for s in (1, 2, 3)]
    print(f"  {CNAME[kind]:<15} s1 {vals[0]:5.1f}  s2 {vals[1]:5.1f}  s3 {vals[2]:5.1f}")
print("\nMean relative drop over all corruptions (mean ± std over folds):")
for me in METHODS:
    print(f"  {LABEL[me]:<20} {100*rob.loc[me, ('mean_rel_drop', 'mean')]:.1f}% ± {100*rob.loc[me, ('mean_rel_drop', 'std')]:.1f}")
print("\nvs baseline (relative drop, lower = more robust):")
for _, r in rob_paired[rob_paired.metric == "mean_rel_drop"].iterrows():
    print(f"  {LABEL[r.method]:<20} {r.mean_diff_pp:+.1f} pp, more robust on {r.folds_better} folds, p_holm={r.p_holm:.3g}")
print("\nNatural conditions, macro-F1 (%) mean over all 20 models:")
for col, d in nat.items():
    g = d.groupby("bucket", observed=True).macro_f1.mean().reindex(test[col].cat.categories)
    print(f"  {col:<18} " + "  ".join(f"{b} {100*v:.1f}" for b, v in g.items()))
print("\nFiles in", OUT)
for f in sorted(OUT.iterdir()):
    print("  ", f.name)

Mean macro-F1 over all 20 models:
  clean                 77.7
  Blur            s1  74.2  s2  63.2  s3  37.6
  Darker          s1  77.4  s2  76.4  s3  74.3
  Brighter        s1  75.5  s2  72.0  s3  67.3
  Noise           s1  72.1  s2  51.7  s3  29.4
  JPEG            s1  73.5  s2  67.5  s3  52.9
  Low resolution  s1  71.8  s2  58.4  s3  43.0

Mean relative drop over all corruptions (mean ± std over folds):
  None (baseline)      17.6% ± 1.8
  Weighted sampler     20.0% ± 2.5
  Class-weighted loss  18.3% ± 0.8
  Oversampling         18.4% ± 2.5

vs baseline (relative drop, lower = more robust):
  Weighted sampler     +2.4 pp, more robust on 0/5 folds, p_holm=0.33
  Class-weighted loss  +0.7 pp, more robust on 1/5 folds, p_holm=0.516
  Oversampling         +0.8 pp, more robust on 2/5 folds, p_holm=0.516

Natural conditions, macro-F1 (%) mean over all 20 models:
  brightness bucket  dark 76.1  medium 76.5  bright 79.4
  sharpness bucket   blurry 79.0  medium 77.8  sharp 73.4

Files in /h